In [ ]:
import os, pathlib, tarfile, shutil, sys, subprocess, json
base='/kaggle/input/datasets/asoksci/eeg-dementia-coursach-full'
out='/kaggle/working/repo'
os.makedirs(out, exist_ok=True)
for tar in pathlib.Path(base).rglob('*.tar'):
    try:
        tf=tarfile.open(tar); tf.extractall(out); tf.close()
    except: pass
for p in ['data','configs','src','reports','figures']:
    s=pathlib.Path(base)/p
    if s.exists(): shutil.copytree(s, pathlib.Path(out)/p, dirs_exist_ok=True)
for d in list(pathlib.Path(out).rglob('src')):
    if str(d)==out+'/src': continue
    for f in d.glob('*.py'):
        dest=pathlib.Path(out)/'src'/f.name
        if f.resolve()!=dest.resolve(): shutil.copy(f, dest)
os.chdir(out)
print('=== LEAD full: download raw + preprocess ===')
!pip install -q openneuro-py mne h5py torch einops -q
!mkdir -p data/raw
!python -m openneuro download --dataset ds004504 --target-dir data/raw/ds004504 --tag 1.0.8
!python -m openneuro download --dataset ds006036 --target-dir data/raw/ds006036 --tag 1.0.6
!ls -lh data/raw/ds004504/sub-001/eeg | head
!ls -lh data/raw/ds006036/sub-001/eeg | head
!git clone https://github.com/DL4mHealth/LEAD.git
!pip install -r LEAD/requirements.txt -q
!huggingface-cli download YiheWang/LEAD --local-dir ./LEAD_ckpt || pip install -q huggingface_hub && python -c "from huggingface_hub import snapshot_download; snapshot_download('YiheWang/LEAD', local_dir='LEAD_ckpt')"
!ls -lh LEAD_ckpt | head
!python src/08_lead_preprocessing.py data/raw/ds004504 Processed/L400/ADFTD-RS
!python src/08_lead_preprocessing.py data/raw/ds006036 Processed/L400/ADFTD-PS
!python src/09_lead_merge.py Processed/L400/ADFTD-RS Processed/L400/ADFTD-PS Processed/L400/ADFTD
!ls -lh Processed/L400/ADFTD/ | head
print('=== finetune ===')
!python LEAD/run.py --task_name finetune --model LEAD --data ADFTD --root_path ./Processed --data_path L400 --features M --seq_len 400 --enc_in 19 --cross_val loso --train_epochs 5 --batch_size 32 --learning_rate 0.0001 --use_gpu True --devices 0 | tee reports/lead_loso.log
print('done')
